In [ ]:
# hide
from typing import Iterator
import numpy as np
import matplotlib.pyplot as plt
import pyquist as pq
from pyquist.helper import frequency_to_pitch, pitch_to_frequency

In [ ]:
# collapse
def iter_frames(audio: pq.Audio, N_H: int, N_F: int) -> Iterator[np.ndarray]:
    for start in range(0, len(audio) - N_F + 1, N_H):
        yield audio.samples[start:start + N_F]


def stft(audio: pq.Audio, N_F: int, N_H: int) -> tuple[np.ndarray, np.ndarray]:
    """The magnitude spectrum of every frame, shape (num_frames, N_F + 1), and the bin frequencies.

    Frame k is centered on time k * N_H / sample_rate. Each windowed frame is zero-padded to 2 * N_F
    samples before the DFT, which samples its spectrum on a finer grid of frequencies.
    """
    padded = pq.Audio(np.pad(audio.samples, ((N_F // 2, N_F // 2), (0, 0))), audio.sample_rate)
    frames = np.array(list(iter_frames(padded, N_H, N_F)))[:, :, 0]
    S = np.abs(np.fft.rfft(frames * np.hanning(N_F), n=2 * N_F, axis=1))
    S[:, 0] = 0.0                                         # ignore 0 Hz, which has no pitch
    return S, np.fft.rfftfreq(2 * N_F, 1 / audio.sample_rate)


def mel_filterbank(freqs: np.ndarray, num_bands: int, f_min: float, f_max: float) -> tuple[np.ndarray, np.ndarray]:
    """Triangular filters evenly spaced on the mel scale, shape (num_bands, len(freqs)), and their centers in Hz."""
    mel_min, mel_max = 2595 * np.log10(1 + np.array([f_min, f_max]) / 700)
    edges = 700 * (10 ** (np.linspace(mel_min, mel_max, num_bands + 2) / 2595) - 1)   # mel back to Hz
    lower, center, upper = edges[:-2, None], edges[1:-1, None], edges[2:, None]
    rising, falling = (freqs - lower) / (center - lower), (upper - freqs) / (upper - center)
    return np.maximum(0.0, np.minimum(rising, falling)), edges[1:-1]


def pitch_spectrogram(S: np.ndarray, freqs: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """A mel spectrogram over the piano's range, with one row per pitch, shape (num_frames, num_pitches).

    Each mel band is labeled with the MIDI pitch nearest its center frequency. We use enough bands that
    nearly every piano key gets one, and where several bands share a pitch we keep the loudest.
    """
    filters, centers = mel_filterbank(freqs, num_bands=512, f_min=27.5, f_max=4186.0)
    mel = S @ filters.T
    band_pitches = np.round(frequency_to_pitch(centers))
    pitches = np.unique(band_pitches).astype(int)
    return np.stack([mel[:, band_pitches == p].max(axis=1) for p in pitches], axis=1), pitches


def notes_from_frames(on: np.ndarray, pitches: np.ndarray, seconds_per_frame: float) -> pq.Score:
    """Turn a (num_frames, num_pitches) on/off grid into a score: each run of "on" frames is one note."""
    events = []
    for j, pitch in enumerate(pitches):
        changes = np.diff(np.concatenate([[0], on[:, j].astype(int), [0]]))   # +1 at a note's start, -1 after it
        for start, end in zip(np.flatnonzero(changes == 1), np.flatnonzero(changes == -1)):
            duration = (end - start) * seconds_per_frame
            events.append((start * seconds_per_frame, {"pitch": int(pitch), "duration": duration}))
    return pq.Score(sorted(events, key=lambda event: event[0]))


def tone(pitch: int, duration: float, sample_rate: int = 44100, **kwargs) -> pq.Audio:
    """A soft tone with three harmonics and 10 ms fades, for listening to a transcription."""
    t = np.arange(int(duration * sample_rate)) / sample_rate
    f0 = pitch_to_frequency(pitch)
    x = sum(np.sin(2 * np.pi * h * f0 * t) / h for h in [1, 2, 3])
    fade = np.clip(np.minimum(t, duration - t) / 0.01, 0.0, 1.0)
    return pq.Audio(fade * x, sample_rate)


def sonify(score: pq.Score, original: pq.Audio) -> pq.Audio:
    """Render a transcription with tone(), at the same length and loudness (RMS) as the original."""
    out = pq.Audio.zeros(len(original), 1, original.sample_rate)
    if len(score) == 0:
        return out
    rendered = score.render(tone)[:len(original)]
    out[:len(rendered)] += rendered
    return out * np.sqrt(np.mean(original.samples ** 2) / np.mean(out.samples ** 2))


def plot_transcriptions(S: np.ndarray, freqs: np.ndarray, seconds_per_frame: float, scores: dict[str, pq.Score]) -> None:
    """One piano roll per score (red bars) over the pitch spectrogram (gray: darker is louder)."""
    P, pitches = pitch_spectrogram(S, freqs)
    t = np.arange(len(P)) * seconds_per_frame
    fig, axes = plt.subplots(len(scores), 1, figsize=(12, 3.8 * len(scores)), sharex=True, squeeze=False)
    for ax, (label, score) in zip(axes[:, 0], scores.items()):
        ax.pcolormesh(t, pitches, 20 * np.log10(P.T / P.max() + 1e-6), cmap="Greys", vmin=-50, vmax=10,
                      shading="nearest")
        ax.barh([e.kwargs["pitch"] for e in score], [e.kwargs["duration"] for e in score],
                left=[e.time for e in score], height=1.0, color="#C41230")
        ax.set_xlim(0, t[-1])
        ax.set_ylim(20.5, 108.5)
        ax.set_yticks(range(24, 109, 12))
        ax.set_ylabel("MIDI pitch", fontsize=12)
        if label:
            ax.text(0.01, 0.95, label, transform=ax.transAxes, fontsize=12, va="top",
                    bbox=dict(facecolor="white", edgecolor="none", alpha=0.8))
    axes[-1, 0].set_xlabel("Time (s)", fontsize=12)
    fig.tight_layout()
    plt.show()

In [ ]:
# Strategy 1 (Chapter 10): the loudest frequency in each frame, rounded to the nearest pitch
audio = pq.Audio.from_file("../assets/audio-trio.wav")
pq.play(audio)                                            # what we are transcribing
N_F, N_H = 8192, 1024                                     # long frames: the bass notes are low
S, freqs = stft(audio, N_F, N_H)
loudest = np.round(frequency_to_pitch(freqs[np.argmax(S, axis=1)]))
loudest[S.max(axis=1) < 0.05 * S.max()] = -1             # too quiet: no pitch
pitches = np.arange(21, 109)                              # the 88 piano keys
on = loudest[:, None] == pitches                          # (num_frames, 88): at most one pitch on per frame
score = notes_from_frames(on, pitches, N_H / audio.sample_rate)   # a new note whenever the pitch changes
print(f"{len(score)} notes")
plot_transcriptions(S, freqs, N_H / audio.sample_rate, {"": score})
pq.play(sonify(score, audio))                             # the transcription alone
pq.play(audio + 0.5 * sonify(score, audio))               # ... and on top of the original

In [ ]:
# Strategy 2: a pitch is on in a frame if its bin is among the loudest K% of all bins
P, pitches = pitch_spectrogram(S, freqs)                  # mel spectrogram, (num_frames, num_pitches)
scores = {}
for K in [1, 10]:
    on = P > np.percentile(P, 100 - K)                    # one global threshold for every bin
    score = notes_from_frames(on, pitches, N_H / audio.sample_rate)   # each run of "on" frames is a note
    print(f"K = {K}%: {len(score)} notes")
    scores[f"K = {K}%"] = score
plot_transcriptions(S, freqs, N_H / audio.sample_rate, scores)
for score in scores.values():
    pq.play(sonify(score, audio))